# 02 · ETL Bronze → Silver — Harmonização de schemas (PySpark)
**Tech Challenge Fase 3 · State of Data Brasil (Data Hackers/Bain)**

| Item | Descrição |
|---|---|
| **Objetivo** | Aplicar o **de-para versionado de colunas** entre as 6 edições, normalizar categorias e derivar variáveis analíticas, unificando tudo em tabelas Silver (Parquet). |
| **Origem** | `bronze/ano=YYYY/*.csv` |
| **Destino** | `silver/silver_core` (2023–2025/26) · `silver/silver_serie_longa` (2019–2025/26) |
| **Requisitos atendidos** | R4 (ETL com Glue Jobs — ver `glue_jobs/`), R5 (camadas), R7 (Spark) |

### Premissas versionadas aplicadas neste notebook
| # | Premissa |
|---|---|
| **P1** | Núcleo obrigatório = 2023/2024/2025-26; 2019/2021/2022 apenas em séries longas (decisão do Ricardo, regra 13.9). |
| **P2** | Dados anonimizados na origem; análise exclusivamente agregada (sem supressão adicional — decisão do Ricardo). |
| **P3** | Salário = ponto médio da faixa: `(low+high)/2`; `Menos de X → X/2`; `Acima de X → X·1,125`. |
| **P4** | Correção de typo na base 2025/26 (`a R$ 3000/mês` → R$ 30.000, se teto < piso ⇒ teto×10). |
| **P5** | Cargos agrupados por função (`cargo_grupo`) para comparabilidade entre edições. |
| **P6** | Multirresposta: % sempre sobre respondentes válidos da questão, nunca sobre o total. |
| **P7** | Booleanos heterogêneos (`1/0`, `True/False`, `TRUE/FALSE`) normalizados para `1/0`. |
| **P8** | Recortes com **n < 30** não são exibidos em gráficos/tabelas — corte de *exibição*, não de processamento: nenhum registro sai do pipeline. Motivo medido: a mediana de faixa é instável em grupo pequeno (com n = 9 ela oscila 7 faixas sob reamostragem). Afeta 1,0% da base. Detalhe em `../config/versioned_assumptions.md`. |

> ⚠️ **Nota de harmonização:** a categoria de senioridade `Especialista/Staff+` existe apenas em 2025/26;
> comparações de senioridade entre anos devem citar essa quebra de série.

> ⚠️ **Nota de reorganização (padrão Medallion).**
>
> O **código** das células abaixo já aponta para a estrutura atual do repositório:
> `../../datalake/{bronze,silver,gold}`, `../../consumption/charts` e as tabelas Gold
> renomeadas em inglês (`gold_roles`, `gold_salary_by_seniority`, …).
>
> **Atualização (28/08/2026):** as saídas abaixo já foram regravadas por reexecução real
> (PySpark 3.5.1 + JDK 17), com a nomenclatura atual — não são mais as saídas originais
> anteriores à reorganização.


In [1]:
# ============================================================
# PORTABILIDADE AWS GLUE (descomente APENAS no Glue Notebook)
# ============================================================
# %glue_version 4.0
# %worker_type G.1X
# %number_of_workers 2
# %idle_timeout 30
# No AWS, ajuste BASE para: s3://<seu-bucket>/datalake
# ============================================================

In [2]:
# -*- coding: utf-8 -*-
"""
ETL Bronze -> Silver | State of Data Brasil
Harmonização de schemas (de-para) das edições 2019/2021/2022 (histórico)
e 2023/2024/2025-26 (núcleo obrigatório).
"""
import json, re, os, shutil
from pyspark.sql import SparkSession, functions as F, types as T

BASE = "../../datalake"
BRONZE, SILVER = f"{BASE}/bronze", f"{BASE}/silver"

In [3]:
# ------------------------------------------------------------------
# 0. Constantes de volumetria (a ingestão ocorre no notebook 01)

In [4]:
# ------------------------------------------------------------------
# A ingestão (cópia dos CSVs brutos para bronze/ano=YYYY/) roda no notebook 01 — não repetida
# aqui para não haver dois donos da mesma responsabilidade (e para não depender de um caminho
# de origem que só existe na máquina de quem já rodou o notebook 01 antes).
VOLUMETRIA_ESPERADA = {2019: 1765, 2021: 2645, 2022: 4271, 2023: 5293, 2024: 5217, 2025: 3495}

spark = (SparkSession.builder.master("local[2]").appName("etl_bronze_silver")
         .config("spark.driver.memory", "3g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

def ler_bronze(ano):
    df = (spark.read.option("header", True).option("escape", '"')
          .csv(f"{BRONZE}/ano={ano}/state_of_data_{ano}.csv"))
    n = df.count()
    assert n == VOLUMETRIA_ESPERADA[ano], f"{ano}: {n} != {VOLUMETRIA_ESPERADA[ano]}"
    print(f"[bronze] {ano}: {n} linhas x {len(df.columns)} colunas")
    return df

26/08/28 11:03:33 WARN Utils: Your hostname, lgmRicardos-MacMini.local resolves to a loopback address: 127.0.0.1; using 192.168.0.31 instead (on interface en1)
26/08/28 11:03:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/28 11:04:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# ------------------------------------------------------------------
# 1. DE-PARA núcleo (2023 / 2024 / 2025)  destino -> origem

In [6]:
# ------------------------------------------------------------------
DEPARA_2025 = {
    "id": "0.a_token", "idade": "1.a_idade", "faixa_idade": "1.a.1_faixa_idade",
    "genero": "1.b_genero", "cor_raca": "1.c_cor/raca/etnia", "pcd": "1.d_pcd",
    "uf": "1.i.1_uf_onde_mora", "regiao": "1.i.2_regiao_onde_mora",
    "nivel_ensino": "1.l_nivel_de_ensino", "area_formacao": "1.m_área_de_formação",
    "situacao_trabalho": "2.a_situação_de_trabalho", "setor": "2.b_setor",
    "num_funcionarios": "2.c_numero_de_funcionarios", "gestor": "2.d_atua_como_gestor",
    "cargo_gestor": "2.e_cargo_como_gestor", "cargo": "2.f_cargo_atual",
    "nivel": "2.g_nivel", "faixa_salarial": "2.h_faixa_salarial",
    "tempo_exp_dados": "2.i_tempo_de_experiencia_em_dados",
    "satisfeito": "2.k_satisfeito_atualmente",
    "entrevistas_6m": "2.m_participou_de_entrevistas_ultimos_6m",
    "mudar_emprego_6m": "2.n_planos_de_mudar_de_emprego_6m",
    "layoff": "2.p_empresa_passou_por_layoff_em_2025",
    "modelo_atual": "2.q_modelo_de_trabalho_atual",
    "modelo_ideal": "2.r_modelo_de_trabalho_ideal",
    "atitude_presencial": "2.s_atitude_em_caso_de_retorno_presencial",
    "prioridade_ia": "3.e_ai_generativa_e_llm_é_uma_prioridade?",
    "atuacao": "4.a.1_atuacao_em_dados",
    "lang_sql": "4.c.1_SQL", "lang_r": "4.c.2_R", "lang_python": "4.c.3_Python",
    "cloud_aws": "4.e.1_Amazon Web Services (AWS)", "cloud_gcp": "4.e.2_Google Cloud (GCP)",
    "cloud_azure": "4.e.3_Azure (Microsoft)",
    "cloud_onprem": "4.e.6_Servidores On Premise/Não utilizamos Cloud",
    "bi_powerbi": "4.g.1_Microsoft PowerBI", "bi_qlik": "4.g.2_Qlik View/Qlik Sense",
    "bi_tableau": "4.g.3_Tableau", "bi_metabase": "4.g.4_Metabase",
    "bi_looker_studio": "4.g.8_Looker Studio(Google Data Studio)",
    "genai_nao_uso": "4.j.1 Não uso soluções de AI Generativa com foco em produtividade",
    "genai_gratuito": "4.j.2 Uso soluções gratuitas de AI Generativa com foco em produtividade",
    "genai_pago_proprio": "4.j.3 Uso e pago pelas soluções de AI Generativa com foco em produtividade",
    "genai_pago_empresa": "4.j.4 A empresa que trabalho paga pelas soluções de AI Generativa com foco em produtividade",
    "genai_copilot": "4.j.5 Uso soluções do tipo Copilot",
    "crit_salario": "2.o.1_Remuneração/Salário", "crit_beneficios": "2.o.2_Benefícios",
    "crit_proposito": "2.o.3_Propósito do trabalho e da empresa",
    "crit_flex_remoto": "2.o.4_Flexibilidade de trabalho remoto",
    "crit_ambiente": "2.o.5_Ambiente e clima de trabalho",
    "crit_aprendizado": "2.o.6_Oportunidade de aprendizado e trabalhar com referências",
    "crit_carreira": "2.o.7_Plano de carreira e oportunidades de crescimento",
    "crit_maturidade": "2.o.8_Maturidade da empresa em termos de tecnologia e dados",
    "crit_gestores": "2.o.9_Qualidade dos gestores e líderes",
    "crit_reputacao": "2.o.10_Reputação que a empresa tem no mercado",
    "des_contratar": "3.d.1_Contratar talentos", "des_reter": "3.d.2_Reter talentos",
    "des_investimentos": "3.d.3_Convencer a empresa a aumentar investimentos",
    "des_remoto": "3.d.4_Gestão de equipes no ambiente remoto",
    "des_multidisciplinar": "3.d.5_Gestão de projetos envolvendo áreas multidisciplinares",
    "des_qualidade": "3.d.6_Organizar as informações com qualidade e confiabilidade",
    "des_volume": "3.d.7_Processar e armazenar um alto volume de dados",
    "des_valor": "3.d.8_Gerar valor para as áreas de negócios",
    "des_ml_prod": "3.d.9_Desenvolver e manter modelos Machine Learning em produção",
    "des_expectativas": "3.d.10_Gerenciar a expectativa das áreas",
    "des_manutencao": "3.d.11_Garantir a manutenção dos projetos e modelos em produção",
    "des_inovacao": "3.d.12_Conseguir levar inovação para a empresa",
    "des_roi": "3.d.13_Garantir (ROI) em projetos de dados",
    "des_tempo": "3.d.14_Dividir o tempo entre entregas técnicas e gestão",
}

# 2024: mesma notação de 2025, com deslocamentos pontuais de código
DEPARA_2024 = dict(DEPARA_2025)
DEPARA_2024.update({
    "layoff": "2.q_empresa_passou_por_layoff_em_2024",
    "modelo_atual": "2.r_modelo_de_trabalho_atual",
    "modelo_ideal": "2.s_modelo_de_trabalho_ideal",
    "atitude_presencial": "2.t_atitude_em_caso_de_retorno_presencial",
    "lang_sql": "4.d.1_SQL", "lang_r": "4.d.2_R", "lang_python": "4.d.3_Python",
    "cloud_aws": "4.h.1_Amazon Web Services (AWS)", "cloud_gcp": "4.h.2_Google Cloud (GCP)",
    "cloud_azure": "4.h.3_Azure (Microsoft)",
    "cloud_onprem": "4.h.6_Servidores On Premise/Não utilizamos Cloud",
    "bi_powerbi": "4.j.1_Microsoft PowerBI", "bi_qlik": "4.j.2_Qlik View/Qlik Sense",
    "bi_tableau": "4.j.3_Tableau", "bi_metabase": "4.j.4_Metabase",
    "bi_looker_studio": "4.j.8_Looker Studio(Google Data Studio)",
    "genai_nao_uso": "4.m.1 Não uso soluções de AI Generativa com foco em produtividade",
    "genai_gratuito": "4.m.2 Uso soluções gratuitas de AI Generativa com foco em produtividade",
    "genai_pago_proprio": "4.m.3 Uso e pago pelas soluções de AI Generativa com foco em produtividade",
    "genai_pago_empresa": "4.m.4 A empresa que trabalho paga pelas soluções de AI Generativa com foco em produtividade",
    "genai_copilot": "4.m.5 Uso soluções do tipo Copilot",
})

DEPARA_2023 = {
    "id": "('P0', 'id')", "idade": "('P1_a ', 'Idade')", "faixa_idade": "('P1_a_1 ', 'Faixa idade')",
    "genero": "('P1_b ', 'Genero')", "cor_raca": "('P1_c ', 'Cor/raca/etnia')", "pcd": "('P1_d ', 'PCD')",
    "uf": "('P1_i_1 ', 'uf onde mora')", "regiao": "('P1_i_2 ', 'Regiao onde mora')",
    "nivel_ensino": "('P1_l ', 'Nivel de Ensino')", "area_formacao": "('P1_m ', 'Área de Formação')",
    "situacao_trabalho": "('P2_a ', 'Qual sua situação atual de trabalho?')", "setor": "('P2_b ', 'Setor')",
    "num_funcionarios": "('P2_c ', 'Numero de Funcionarios')", "gestor": "('P2_d ', 'Gestor?')",
    "cargo_gestor": "('P2_e ', 'Cargo como Gestor')", "cargo": "('P2_f ', 'Cargo Atual')",
    "nivel": "('P2_g ', 'Nivel')", "faixa_salarial": "('P2_h ', 'Faixa salarial')",
    "tempo_exp_dados": "('P2_i ', 'Quanto tempo de experiência na área de dados você tem?')",
    "satisfeito": "('P2_k ', 'Você está satisfeito na sua empresa atual?')",
    "entrevistas_6m": "('P2_m ', 'Você participou de entrevistas de emprego nos últimos 6 meses?')",
    "mudar_emprego_6m": "('P2_n ', 'Você pretende mudar de emprego nos próximos 6 meses?')",
    "layoff": "('P2_q ', 'Empresa que trabaha passou por layoff em 2023')",
    "modelo_atual": "('P2_r ', 'Atualmente qual a sua forma de trabalho?')",
    "modelo_ideal": "('P2_s ', 'Qual a forma de trabalho ideal para você?')",
    "atitude_presencial": "('P2_t ', 'Caso sua empresa decida pelo modelo 100% presencial qual será sua atitude?')",
    "prioridade_ia": "('P3_e ', 'AI Generativa é uma prioridade em sua empresa?')",
    "atuacao": "('P4_a_1 ', 'Atuacao')",
    "lang_sql": "('P4_d_1 ', 'SQL')", "lang_r": "('P4_d_2 ', 'R ')", "lang_python": "('P4_d_3 ', 'Python')",
    "cloud_aws": "('P4_h_2 ', 'Amazon Web Services (AWS)')", "cloud_gcp": "('P4_h_3 ', 'Google Cloud (GCP)')",
    "cloud_azure": "('P4_h_1 ', 'Azure (Microsoft)')",
    "cloud_onprem": "('P4_h_6 ', 'Servidores On Premise/Não utilizamos Cloud')",
    "bi_powerbi": "('P4_j_1 ', 'Microsoft PowerBI')", "bi_qlik": "('P4_j_2 ', 'Qlik View/Qlik Sense')",
    "bi_tableau": "('P4_j_3 ', 'Tableau')", "bi_metabase": "('P4_j_4 ', 'Metabase')",
    "bi_looker_studio": "('P4_j_8 ', 'Looker Studio(Google Data Studio)')",
    "genai_nao_uso": "('P4_m_1 ', 'Não uso soluções de AI Generativa com foco em produtividade')",
    "genai_gratuito": "('P4_m_2 ', 'Uso soluções gratuitas de AI Generativa com foco em produtividade')",
    "genai_pago_proprio": "('P4_m_3 ', 'Uso e pago pelas soluções de AI Generativa com foco em produtividade')",
    "genai_pago_empresa": "('P4_m_4 ', 'A empresa que trabalho paga pelas soluções de AI Generativa com foco em produtividade')",
    "genai_copilot": "('P4_m_5 ', 'Uso soluções do tipo Copilot')",
    "crit_salario": "('P2_o_1 ', 'Remuneração/Salário')", "crit_beneficios": "('P2_o_2 ', 'Benefícios')",
    "crit_proposito": "('P2_o_3 ', 'Propósito do trabalho e da empresa')",
    "crit_flex_remoto": "('P2_o_4 ', 'Flexibilidade de trabalho remoto')",
    "crit_ambiente": "('P2_o_5 ', 'Ambiente e clima de trabalho')",
    "crit_aprendizado": "('P2_o_6 ', 'Oportunidade de aprendizado e trabalhar com referências na área')",
    "crit_carreira": "('P2_o_7 ', 'Plano de carreira e oportunidades de crescimento profissional')",
    "crit_maturidade": "('P2_o_8 ', 'Maturidade da empresa em termos de tecnologia e dados')",
    "crit_gestores": "('P2_o_9 ', 'Qualidade dos gestores e líderes')",
    "crit_reputacao": "('P2_o_10 ', 'Reputação que a empresa tem no mercado')",
    "des_contratar": "('P3_d_1 ', 'a Contratar novos talentos.')",
    "des_reter": "('P3_d_2 ', 'b Reter talentos.')",
    "des_investimentos": "('P3_d_3 ', 'c Convencer a empresa a aumentar os investimentos na área de dados.')",
    "des_remoto": "('P3_d_4 ', 'd Gestão de equipes no ambiente remoto.')",
    "des_multidisciplinar": "('P3_d_5 ', 'e Gestão de projetos envolvendo áreas multidisciplinares da empresa.')",
    "des_qualidade": "('P3_d_6 ', 'f Organizar as informações e garantir a qualidade e confiabilidade.')",
    "des_volume": "('P3_d_7 ', 'g Conseguir processar e armazenar um alto volume de dados.')",
    "des_valor": "('P3_d_8 ', 'h Conseguir gerar valor para as áreas de negócios através de estudos e experimentos.')",
    "des_ml_prod": "('P3_d_9 ', 'i Desenvolver e manter modelos Machine Learning em produção.')",
    "des_expectativas": "('P3_d_10 ', 'j Gerenciar a expectativa das áreas de negócio em relação as entregas das equipes de dados.')",
    "des_manutencao": "('P3_d_11 ', 'k Garantir a manutenção dos projetos e modelos em produção, em meio ao crescimento da empresa.')",
    "des_inovacao": "('P3_d_12 ', 'Conseguir levar inovação para a empresa através dos dados.')",
    "des_roi": "('P3_d_13 ', 'Garantir retorno do investimento (ROI) em projetos de dados.')",
    "des_tempo": "('P3_d_14 ', 'Dividir o tempo entre entregas técnicas e gestão.')",
}

In [7]:
# ------------------------------------------------------------------
# 2. DE-PARA histórico (2019/2021/2022) — série longa mínima

In [8]:
# ------------------------------------------------------------------
DEPARA_HIST = {
    2019: {"genero": "('P2', 'gender')",
           "lang_sql": "('P21', 'sql_')", "lang_r": "('P21', 'r')", "lang_python": "('P21', 'python')",
           "cloud_aws": "('P25', 'aws')", "cloud_gcp": "('P25', 'gcp')", "cloud_azure": "('P25', 'azure')"},
    2021: {"genero": "('P1_b ', 'Genero')",
           "lang_sql": "('P4_d_a ', 'SQL')", "lang_r": "('P4_d_b ', 'R ')", "lang_python": "('P4_d_c ', 'Python')",
           "cloud_aws": "('P4_g_a ', 'Amazon Web Services (AWS)')",
           "cloud_gcp": "('P4_g_b ', 'Google Cloud (GCP)')", "cloud_azure": "('P4_g_c ', 'Azure (Microsoft)')"},
    2022: {"genero": "('P1_b ', 'Genero')",
           "lang_sql": "('P4_d_1 ', 'SQL')", "lang_r": "('P4_d_2 ', 'R ')", "lang_python": "('P4_d_3 ', 'Python')",
           "cloud_aws": "('P4_h_2 ', 'Amazon Web Services (AWS)')",
           "cloud_gcp": "('P4_h_3 ', 'Google Cloud (GCP)')", "cloud_azure": "('P4_h_1 ', 'Azure (Microsoft)')"},
}

In [9]:
# ------------------------------------------------------------------
# 3. Funções de harmonização (UDFs)

In [10]:
# ------------------------------------------------------------------
def parse_salario_pm(faixa):
    """Ponto médio da faixa salarial (Premissa P3).
    - 'de R$ X a R$ Y'  -> (X+Y)/2
    - 'Menos de R$ X'   -> X/2
    - 'Acima de R$ X'   -> X * 1.125
    - Correção de typo: se Y < X, multiplica Y por 10 (ex.: 'R$ 3000' -> 30.000)."""
    if faixa is None:
        return None
    nums = [float(x.replace(".", "")) for x in re.findall(r"\d[\d\.]*", faixa)]
    if not nums:
        return None
    low = nums[0]
    if "Menos" in faixa:
        return low / 2.0
    if "Acima" in faixa:
        return low * 1.125
    if len(nums) >= 2:
        high = nums[1]
        if high < low:          # typo conhecido na base 2025 ('a R$ 3000/mês')
            high = high * 10
        return (low + high) / 2.0
    return low

def harmoniza_cargo(cargo):
    """Agrupa nomenclaturas de cargo divergentes entre edições (Premissa P5)."""
    if cargo is None:
        return None
    c = cargo.lower()
    if "outra" in c: return "Outros"
    if "engenheiro de dados" in c or "data engineer" in c or "arquiteto" in c: return "Engenharia/Arquitetura de Dados"
    if "analista de dados" in c or "data analyst" in c: return "Análise de Dados"
    if "cientista de dados" in c or "data scientist" in c: return "Ciência de Dados"
    if "analista de bi" in c or "bi analyst" in c: return "Business Intelligence"
    if "analytics engineer" in c: return "Analytics Engineer"
    if "machine learning" in c or "ml engineer" in c or "ai engineer" in c: return "ML/AI Engineer"
    if "negócios" in c or "business analyst" in c: return "Análise de Negócios"
    if "desenvolvedor" in c or "software" in c or "sistemas" in c: return "Eng. de Software"
    if "product" in c or "produto" in c: return "Produto (DPM/PM)"
    if "dba" in c or "administrador de banco" in c: return "DBA"
    if "estatístico" in c or "economista" in c: return "Estatística/Economia"
    if "professor" in c or "pesquisador" in c: return "Professor/Pesquisador"
    if "suporte" in c or "analista de negócios" in c: return "Outros"
    return "Outros"

def harmoniza_modelo(m):
    if m is None: return None
    ml = m.lower()
    if "100% remoto" in ml: return "100% remoto"
    if "100% presencial" in ml: return "100% presencial"
    if "dias fixos" in ml: return "Híbrido (dias fixos)"
    if "flexível" in ml or "flexivel" in ml: return "Híbrido flexível"
    return "Outro"

def harmoniza_prioridade_ia(p):
    if p is None: return None
    if p.startswith("Sim, é nossa principal"): return "1. Principal prioridade da empresa"
    if p.startswith("Sim, está entre"): return "2. Entre as principais (2-4 anos)"
    if p.startswith("Mais ou menos"): return "3. Iniciativas isoladas, sem foco"
    if p.startswith("Não é uma iniciativa"): return "4. Não é prioridade"
    if p.startswith("Não sei"): return "5. Não sabe opinar"
    return "5. Não sabe opinar"

udf_salario = F.udf(parse_salario_pm, T.DoubleType())
udf_cargo = F.udf(harmoniza_cargo, T.StringType())
udf_modelo = F.udf(harmoniza_modelo, T.StringType())
udf_prio_ia = F.udf(harmoniza_prioridade_ia, T.StringType())

BINARIAS = [c for c in DEPARA_2025 if c.startswith(("lang_", "cloud_", "bi_", "genai_", "crit_", "des_"))] + ["gestor", "satisfeito"]

def normaliza_binaria(col):
    """'1'/'1.0'/'True' -> 1 ; '0'/'0.0'/'False' -> 0 ; demais -> null."""
    return (F.when(F.upper(F.col(col)).isin("1", "1.0", "TRUE"), F.lit(1))
             .when(F.upper(F.col(col)).isin("0", "0.0", "FALSE"), F.lit(0))
             .otherwise(F.lit(None).cast("int")))

def transforma_nucleo(df, depara, ano):
    # seleção + renomeação (backticks protegem nomes com pontos/aspas/parênteses)
    faltantes = [src for src in depara.values() if src not in df.columns]
    assert not faltantes, f"{ano}: colunas ausentes no CSV: {faltantes}"
    df = df.select([F.col(f"`{src}`").alias(dst) for dst, src in depara.items()])
    df = df.withColumn("ano", F.lit(ano))
    for c in BINARIAS:
        df = df.withColumn(c, normaliza_binaria(c))
    df = (df
          .withColumn("idade", F.col("idade").cast("int"))
          .withColumn("salario_pm", udf_salario("faixa_salarial"))
          .withColumn("cargo_grupo", udf_cargo("cargo"))
          .withColumn("modelo_atual_h", udf_modelo("modelo_atual"))
          .withColumn("modelo_ideal_h", udf_modelo("modelo_ideal"))
          .withColumn("prioridade_ia_h", udf_prio_ia("prioridade_ia"))
          .withColumn("layoff_sim", F.when(F.col("layoff").startswith("Sim"), 1)
                                     .when(F.col("layoff").startswith("Não"), 0)
                                     .otherwise(F.lit(None).cast("int"))))
    return df

def transforma_historico(df, depara, ano):
    faltantes = [src for src in depara.values() if src not in df.columns]
    assert not faltantes, f"{ano}: colunas ausentes: {faltantes}"
    df = df.select([F.col(f"`{src}`").alias(dst) for dst, src in depara.items()])
    df = df.withColumn("ano", F.lit(ano))
    for c in ["lang_sql", "lang_r", "lang_python", "cloud_aws", "cloud_gcp", "cloud_azure"]:
        df = df.withColumn(c, normaliza_binaria(c))
    return df

In [11]:
# ------------------------------------------------------------------
# 4. Execução

In [12]:
# ------------------------------------------------------------------
nucleo = None
for ano, depara in [(2023, DEPARA_2023), (2024, DEPARA_2024), (2025, DEPARA_2025)]:
    t = transforma_nucleo(ler_bronze(ano), depara, ano)
    nucleo = t if nucleo is None else nucleo.unionByName(t)

nucleo.write.mode("overwrite").partitionBy("ano").parquet(f"{SILVER}/silver_core")

# Série longa 6 anos: histórico + colunas equivalentes do núcleo
serie_cols = ["ano", "genero", "lang_sql", "lang_r", "lang_python", "cloud_aws", "cloud_gcp", "cloud_azure"]
serie = None
for ano, depara in DEPARA_HIST.items():
    t = transforma_historico(ler_bronze(ano), depara, ano)
    serie = t if serie is None else serie.unionByName(t)
serie = serie.select(serie_cols).unionByName(nucleo.select(serie_cols))
serie.write.mode("overwrite").partitionBy("ano").parquet(f"{SILVER}/silver_serie_longa")

[bronze] 2023: 5293 linhas x 399 colunas


[bronze] 2024: 5217 linhas x 403 colunas


[bronze] 2025: 3495 linhas x 388 colunas


[bronze] 2019: 1765 linhas x 170 colunas
[bronze] 2021: 2645 linhas x 356 colunas


[bronze] 2022: 4271 linhas x 353 colunas


In [13]:
# ------------------------------------------------------------------
# 5. Validação / reconciliação bronze x silver

## 6. EDA e Qualidade dos Dados
Espelho da **Seção 2.5** e da **Tabela 5** do relatório técnico: unicidade de tokens e preenchimento das variáveis-chave, com o motivo estrutural de cada nulo (fluxo condicional do questionário — nenhuma imputação aplicada; denominadores por questão, premissa P6).

In [14]:
# ------------------------------------------------------------------
# 6. EDA e Qualidade dos Dados (espelha a Seção 2.5 e a Tabela 5 do relatório)
# ------------------------------------------------------------------
sc_eda = spark.read.parquet(f"{SILVER}/silver_core")          # núcleo 2023–2025/26
n_core = sc_eda.count()                                        # volumetria do núcleo
print(f"silver_core (núcleo 2023–2025/26): {n_core} linhas")

# --- Unicidade: tokens repetidos por edição (mantidos — impacto estatístico nulo) ---
print("\n== Unicidade de tokens por edição ==")
(sc_eda.groupBy("ano")
    .agg(F.count("*").alias("linhas"), F.countDistinct("id").alias("ids_distintos"))
    .withColumn("tokens_repetidos", F.col("linhas") - F.col("ids_distintos"))
    .orderBy("ano").show())

# --- Preenchimento das variáveis-chave: nulos são ESTRUTURAIS, não falha de coleta ---
MOTIVOS = {
    "genero": "pergunta universal",
    "atuacao": "pergunta universal",
    "faixa_salarial": "apenas quem tem vínculo ativo",
    "modelo_atual": "apenas quem tem vínculo ativo",
    "satisfeito": "apenas quem tem vínculo ativo",
    "nivel": "apenas contribuidores individuais empregados",
    "cargo": "apenas contribuidores individuais empregados",
    "cargo_gestor": "apenas gestores",
    "prioridade_ia": "bloco exclusivo de gestores",
}
linha = sc_eda.select([F.round(100 * F.count(F.col(c)) / n_core, 1).alias(c) for c in MOTIVOS]).collect()[0]
print(f"== Preenchimento das variáveis-chave (% sobre n = {n_core}) ==")
for c, motivo in MOTIVOS.items():
    print(f"{c:16s} {linha[c]:5.1f}%  | nulo estrutural: {motivo}")
print("\nConclusão: nulos estruturais do questionário; nenhuma imputação aplicada (premissa P6).")

silver_core (núcleo 2023–2025/26): 14005 linhas

== Unicidade de tokens por edição ==
+----+------+-------------+----------------+
| ano|linhas|ids_distintos|tokens_repetidos|
+----+------+-------------+----------------+
|2023|  5293|         5293|               0|
|2024|  5217|         5215|               2|
|2025|  3495|         3494|               1|
+----+------+-------------+----------------+



== Preenchimento das variáveis-chave (% sobre n = 14005) ==
genero           100.0%  | nulo estrutural: pergunta universal
atuacao          100.0%  | nulo estrutural: pergunta universal
faixa_salarial    91.7%  | nulo estrutural: apenas quem tem vínculo ativo
modelo_atual      91.7%  | nulo estrutural: apenas quem tem vínculo ativo
satisfeito        91.7%  | nulo estrutural: apenas quem tem vínculo ativo
nivel             72.7%  | nulo estrutural: apenas contribuidores individuais empregados
cargo             72.7%  | nulo estrutural: apenas contribuidores individuais empregados
cargo_gestor      19.1%  | nulo estrutural: apenas gestores
prioridade_ia     18.5%  | nulo estrutural: bloco exclusivo de gestores

Conclusão: nulos estruturais do questionário; nenhuma imputação aplicada (premissa P6).


## 7. Justificativa quantitativa da Premissa P8 (reamostragem bootstrap)

A **Seção 2.6** do relatório técnico e a **Premissa P8** (`../config/versioned_assumptions.md`) afirmam que
recortes com **n < 30** não são exibidos em gráficos/tabelas porque a mediana de uma faixa salarial é
instável nesse tamanho de amostra. Esta célula materializa essa afirmação em código — antes, o número
citado no relatório não tinha origem reproduzível em nenhum notebook.

**Método:** bootstrap não paramétrico. Para cada grupo de cargo, reamostramos `salario_pm` (ponto médio de
faixa, edição 2025/26) **3.000 vezes, com reposição, no mesmo tamanho da amostra original**; calculamos a
mediana de cada reamostragem; tomamos os percentis 2,5% e 97,5% dessa distribuição de 3.000 medianas como
intervalo de 95%. A semente (`numpy.random.default_rng`) é fixa por grupo — qualquer reexecução desta
célula reproduz exatamente os mesmos números.

**Grupos escolhidos:** dois abaixo do corte de P8 (Estatística/Economia, n=9; Professor/Pesquisador, n=16)
e dois acima (Produto DPM/PM, n=33; Análise de Dados, n=599), para contrastar o comportamento nos dois
lados do limiar.

**Nota de honestidade metodológica:** por ser um método estocástico, os limites exatos do intervalo variam
ligeiramente entre execuções com sementes diferentes da usada originalmente ao redigir o relatório — a
mediana em si (que não depende de reamostragem) bate exatamente com a Tabela do §2.6. Já a conclusão
qualitativa do intervalo — grupos pequenos oscilam várias faixas inteiras, grupos grandes são estáveis —
é robusta a qualquer semente, como o resultado abaixo confirma.

In [15]:
# ------------------------------------------------------------------
# 7. Reamostragem bootstrap — justificativa quantitativa da Premissa P8
# ------------------------------------------------------------------
import numpy as np
from decimal import Decimal, ROUND_HALF_UP

def r0(x):
    """Arredonda HALF_UP (mesma convenção do F.round do Spark usado em toda a camada Gold).
    Necessário porque salario_pm sempre termina em ',5' com parte inteira par (ponto médio de
    faixa) — o round-half-to-even nativo do Python/format ',.0f' arredondaria sempre para baixo,
    divergindo em R$1 da Tabela do §2.6 e de versioned_assumptions.md."""
    return int(Decimal(repr(float(x))).quantize(Decimal("1"), rounding=ROUND_HALF_UP))

GRUPOS_P8 = ["Estatística/Economia", "Professor/Pesquisador", "Produto (DPM/PM)", "Análise de Dados"]
SEED = 43
N_RESAMOSTRAGENS = 3000

# collect() em vez de toPandas(): evita depender de distutils (removido no Python 3.12,
# quebra o toPandas() do PySpark 3.5.1) — a base já é pequena (poucas centenas de linhas).
linhas = (sc_eda.filter((F.col("ano") == 2025) & F.col("cargo_grupo").isin(GRUPOS_P8)
                        & F.col("salario_pm").isNotNull())
          .select("cargo_grupo", "salario_pm").collect())
por_grupo = {g: np.array([r.salario_pm for r in linhas if r.cargo_grupo == g]) for g in GRUPOS_P8}

print(f"== Reamostragem bootstrap (seed={SEED}, {N_RESAMOSTRAGENS} reamostragens com reposição) ==\n")
print(f"{'Grupo':26s}{'n':>5s}{'Mediana':>13s}{'IC 95% baixo':>16s}{'IC 95% alto':>15s}")
for i, grp in enumerate(GRUPOS_P8):
    vals = por_grupo[grp]
    n = len(vals)
    rng = np.random.default_rng(SEED + i)          # semente distinta por grupo, fixa e reprodutível
    medianas = np.array([np.median(rng.choice(vals, size=n, replace=True)) for _ in range(N_RESAMOSTRAGENS)])
    lo, hi = np.percentile(medianas, [2.5, 97.5])
    print(f"{grp:26s}{n:5d}  R$ {r0(np.median(vals)):9,d}   R$ {r0(lo):9,d}      R$ {r0(hi):9,d}")

print("\nConclusão: os dois grupos abaixo do corte de P8 (n=9 e n=16) têm intervalo de 95% que atravessa\n"
      "várias faixas salariais inteiras — a mediana não é uma estimativa estável nesse tamanho de amostra.\n"
      "Os dois grupos com n>=30 (33 e 599) têm intervalo muito mais estreito ou nulo. Isso sustenta\n"
      "empiricamente a Premissa P8 — corte de EXIBIÇÃO em n<30, não de processamento\n"
      "(ver '../config/versioned_assumptions.md' e Seção 2.6 do relatório técnico).")

== Reamostragem bootstrap (seed=43, 3000 reamostragens com reposição) ==

Grupo                         n      Mediana    IC 95% baixo    IC 95% alto
Estatística/Economia          9  R$     5,001   R$     1,501      R$    18,001
Professor/Pesquisador        16  R$    10,001   R$     5,001      R$    16,001
Produto (DPM/PM)             33  R$    14,001   R$    10,001      R$    18,113


Análise de Dados            599  R$     7,001   R$     7,001      R$     7,001

Conclusão: os dois grupos abaixo do corte de P8 (n=9 e n=16) têm intervalo de 95% que atravessa
várias faixas salariais inteiras — a mediana não é uma estimativa estável nesse tamanho de amostra.
Os dois grupos com n>=30 (33 e 599) têm intervalo muito mais estreito ou nulo. Isso sustenta
empiricamente a Premissa P8 — corte de EXIBIÇÃO em n<30, não de processamento
(ver '../config/versioned_assumptions.md' e Seção 2.6 do relatório técnico).


In [16]:
# ------------------------------------------------------------------
sc = spark.read.parquet(f"{SILVER}/silver_core")
sl = spark.read.parquet(f"{SILVER}/silver_serie_longa")
print("\n== Reconciliação ==")
sc.groupBy("ano").count().orderBy("ano").show()
sl.groupBy("ano").count().orderBy("ano").show()
for ano in [2023, 2024, 2025]:
    n = sc.filter(F.col("ano") == ano).count()
    assert n == VOLUMETRIA_ESPERADA[ano], f"silver {ano} divergente"
print("Checagem salario_pm (2025):")
sc.filter("ano=2025").select("faixa_salarial", "salario_pm").distinct().orderBy("salario_pm").show(20, False)
print("Checagem prioridade IA:")
sc.groupBy("ano", "prioridade_ia_h").count().orderBy("ano", "prioridade_ia_h").show(30, False)

# exporta o de-para versionado (evidência R4/R5 e regra 13.10)
os.makedirs("../config", exist_ok=True)
with open("../config/column_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"2025": DEPARA_2025, "2024": DEPARA_2024, "2023": DEPARA_2023,
               "historico": {str(k): v for k, v in DEPARA_HIST.items()}}, f, ensure_ascii=False, indent=2)
print("\nOK — silver gravada e de-para versionado.")
spark.stop()


== Reconciliação ==
+----+-----+
| ano|count|
+----+-----+
|2023| 5293|
|2024| 5217|
|2025| 3495|
+----+-----+

+----+-----+
| ano|count|
+----+-----+
|2019| 1765|
|2021| 2645|
|2022| 4271|
|2023| 5293|
|2024| 5217|
|2025| 3495|
+----+-----+



Checagem salario_pm (2025):


+--------------------------------+----------+
|faixa_salarial                  |salario_pm|
+--------------------------------+----------+
|NULL                            |NULL      |
|Menos de R$ 1.000/mês           |500.0     |
|de R$ 1.001/mês a R$ 2.000/mês  |1500.5    |
|de R$ 2.001/mês a R$ 3.000/mês  |2500.5    |
|de R$ 3.001/mês a R$ 4.000/mês  |3500.5    |
|de R$ 4.001/mês a R$ 6.000/mês  |5000.5    |
|de R$ 6.001/mês a R$ 8.000/mês  |7000.5    |
|de R$ 8.001/mês a R$ 12.000/mês |10000.5   |
|de R$ 12.001/mês a R$ 16.000/mês|14000.5   |
|de R$ 16.001/mês a R$ 20.000/mês|18000.5   |
|de R$ 20.001/mês a R$ 25.000/mês|22500.5   |
|de R$ 25.001/mês a R$ 3000/mês  |27500.5   |
|de R$ 25.001/mês a R$ 30.000/mês|27500.5   |
|de R$ 30.001/mês a R$ 40.000/mês|35000.5   |
|Acima de R$ 40.001/mês          |45001.125 |
+--------------------------------+----------+

Checagem prioridade IA:
+----+----------------------------------+-----+
|ano |prioridade_ia_h                   |count|
+----